# Mini project: who will churn?

**Darunz · Data Science**  |  Modules 8 and 9

A telecom company has 500 customers. Some of them left last quarter. Your job:

> **Which customers will churn next, and what should the retention team do about it?**

Work through the nine parts below. Each one has a task list and some starter code with `TODO` in it.
Write your answer to every question in the markdown cell that follows the code, in your own words.

**Dataset:** `telecom-churn.csv`

| Column | Meaning |
|---|---|
| `tenure_months` | How long they have been a customer |
| `monthly_charges` | Their monthly bill in rupees |
| `complaints` | Complaints raised last quarter |
| `churn` | 1 = left, 0 = stayed |

**Before you submit:** Kernel → Restart & Run All. If it errors, it is not finished.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

---
## Part 1: Load and understand the problem

1. Load the CSV.
2. Check the shape, the dtypes, missing values and duplicates.
3. Answer below: is this regression, classification or clustering? What is X and what is y?

In [ ]:
df = pd.read_csv('telecom-churn.csv')

# TODO: shape, head, info
# TODO: df.isna().sum() and df.duplicated().sum()


**Your answer:** _problem type, features, target_

---
## Part 2: Explore before you model

1. What percentage of customers churned? (This decides whether accuracy is a useful metric later.)
2. Plot a histogram of `tenure_months` and of `monthly_charges`.
3. Compare churners and non-churners: `df.groupby('churn').mean()`.
4. Build a correlation matrix and plot the heatmap.
5. Answer below: which feature looks most related to churn, and in which direction?

In [ ]:
# TODO: churn rate
# TODO: two histograms
# TODO: groupby comparison
# TODO: correlation heatmap


**Your answer:** _churn rate, and what the comparison shows_

---
## Part 3: Split, then scale

Order matters. Split first, then fit the scaler on the training rows only.

1. Build `X` (three features) and `y` (churn).
2. Split 70/30 with `random_state=42` and `stratify=y`. Why stratify? Answer below.
3. Scale with `StandardScaler`: `fit_transform` on train, `transform` on test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['tenure_months', 'monthly_charges', 'complaints']]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
# TODO: transform the test set (do NOT fit again)

print(len(X_train), len(X_test))

**Your answer:** _why stratify, and why the scaler is never fitted on test data_

---
## Part 4: Baseline model

1. Fit a `LogisticRegression` on the scaled training data.
2. Predict on the test set, and also get the probabilities with `predict_proba`.
3. Convert the coefficients into odds ratios with `np.exp`.
4. Answer below: explain each of the three odds ratios in one sentence, as if to a manager.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
# TODO: fit
# TODO: pred = ...   and   prob = ...[:, 1]

# TODO: pd.DataFrame({'feature': X.columns, 'coef': ..., 'odds_ratio': np.exp(...)})


**Your answer:** _the three odds ratios in plain business language_

---
## Part 5: Score it properly

1. Print the confusion matrix. Label which number is TP, TN, FP and FN.
2. Print `classification_report`.
3. Calculate accuracy, precision, recall and F1 yourself from the four numbers, and check they match.
4. Print the AUC (pass the probabilities, not the labels).
5. Answer below: a model that predicts 'nobody churns' would score what accuracy on this data? What does that tell you?

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

# TODO: confusion matrix
# TODO: classification report
# TODO: AUC
# TODO: plot the ROC curve


**Your answer:** _which mistake this model makes more often, FP or FN, and why that matters here_

---
## Part 6: Choose a threshold, and defend it

This is the part that matters most. The default 0.5 is not sacred.

Assume the business tells you:

- A retention offer costs **₹300**, whoever it goes to.
- Keeping a customer who would have left is worth **₹4,000**.

1. For thresholds 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, compute precision, recall, F1 and:

   `net_value = 4000 * TP - 300 * (TP + FP)`

2. Put it in one DataFrame, one row per threshold.
3. Answer below: which threshold do you recommend, and why? Would your answer change if an offer cost ₹2,000?

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

rows = []
for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_th = (prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_th).ravel()
    # TODO: append a dict with th, precision, recall, f1, tp, fp, fn and net_value

# TODO: pd.DataFrame(rows)


**Your answer:** _your threshold, the money reasoning, and what changes at ₹2,000 per offer_

---
## Part 7: Compare five models

Same split, same scaled data, so the comparison is fair.

1. Train KNN (k=5), Naive Bayes, a decision tree (`max_depth=3`), a random forest and an SVM.
2. Build a table of train accuracy, test accuracy and test F1 for all six models including logistic regression.
3. Print the tree's rules with `export_text`.
4. Answer below: which model wins on F1? Does any model show 1.0 training accuracy, and what is that called?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    'Logistic':      LogisticRegression(),
    'KNN':           KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':   GaussianNB(),
    'Tree (d=3)':    DecisionTreeClassifier(max_depth=3, random_state=1),
    'Random forest': RandomForestClassifier(n_estimators=200, random_state=1),
    'SVM':           SVC(random_state=1),
}

# TODO: loop, fit each on X_train_s, collect train acc, test acc, test F1
# TODO: show the results as a DataFrame sorted by test F1


**Your answer:** _the winner, and what the training scores tell you_

---
## Part 8: Cross-validate the winner

1. Run `cross_val_score` with `cv=5` on the winning model, scoring `'f1'`.
2. Report the five scores, the mean and the spread.
3. Answer below: is this model stable, or does it depend on which rows it sees?

In [ ]:
from sklearn.model_selection import cross_val_score

# TODO: scale the full X, then cross-validate the winning model


**Your answer:** _mean score, spread, and whether you trust it_

---
## Part 9: Write the recommendation

No code. Write for the retention manager, who has never heard of a confusion matrix.

Cover:

1. **Three findings**, each with the number behind it.
2. **Who to contact**: how many customers, at which threshold, and the expected value.
3. **Two actions** the company should take based on the odds ratios, not just the model.
4. **One limitation**: something this data cannot tell you.

**Your recommendation:**

_Write here._

---
## Before you submit

- [ ] Kernel → Restart & Run All, with no errors
- [ ] Every 'Your answer' cell is filled in
- [ ] Every plot has a title and axis labels
- [ ] No cell just prints a number with no sentence explaining it

**Darunz** · Darun N · Chennai